# ?? 8:?????(Observer Bias)

## 8.6 ??????

??????????,?????????

In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt
import thinkbayes as tb

OBSERVED_GAP_TIMES = [
    428.0, 705.0, 407.0, 465.0, 433.0, 425.0, 204.0, 506.0, 143.0, 351.0, 
    450.0, 598.0, 464.0, 749.0, 341.0, 586.0, 754.0, 256.0, 378.0, 435.0, 
    176.0, 405.0, 360.0, 519.0, 648.0, 374.0, 483.0, 537.0, 578.0, 534.0, 
    577.0, 619.0, 538.0, 331.0, 186.0, 629.0, 193.0, 360.0, 660.0, 484.0, 
    512.0, 315.0, 457.0, 404.0, 740.0, 388.0, 357.0, 485.0, 567.0, 160.0, 
    428.0, 387.0, 901.0, 187.0, 622.0, 616.0, 585.0, 474.0, 442.0, 499.0, 
    437.0, 620.0, 351.0, 286.0, 373.0, 232.0, 393.0, 745.0, 636.0, 758.0,
]

passenger_data = [
    (17, 4.6, 9),
    (22, 1.0, 0),
    (23, 1.4, 4),
    (18, 5.4, 12),
    (4, 5.8, 11),
]


def plot_dist(dist, c=None):
    x, y = zip(*sorted(dist.Items()))
    plt.plot(x, y, c=c)

class TruthGaps(tb.Pmf):
    
    def __init__(self, gaps, x=np.linspace(0.1, 20, 200)):
        super().__init__(x)
        self.kde = gaussian_kde(gaps_min)
        self.make_pmf(x)
        
    def make_pmf(self, x):
        for interval in x:
            self.Set(interval, self.evaluate(interval))
        
        self.Normalize()
        

    def evaluate(self, x):
        return self.kde(x)[0]


def BiasPmf(pmf):
    # pmf ????????
    new_pmf = pmf.Copy()
    
    # ?????,????????????????,x,???????????? x ???
    for x, p in pmf.Items():
        new_pmf.Mult(x, x)
        
    # ???
    new_pmf.Normalize()
    
    #new_pmf ????????
    return new_pmf

def MakeRange(low, high):
    sec = range(low, int(high*60)+1) # ??
    return np.array(sec)/60


def MakeUniformPmf(low, high):
    # low, high ????????(?????)
    pmf = tb.Pmf()
    for x in MakeRange(low, high): # ???????????(??)
        pmf.Set(x, 1)
    
    pmf.Normalize()
    return pmf


def PmfOfWaitTime(pmf_zb):
    
    # metapmf ???????????????
    metapmf = tb.Pmf()
    for gap, prob in pmf_zb.Items():
        uniform = MakeUniformPmf(0, gap) # ??????
        metapmf.Set(uniform, prob)
    
    # ? 5.6 ????,? MakeMixture ?????
    pmf_y = tb.MakeMixture(metapmf)
        
    return pmf_y


class WaitTimeCalculator(object):
    
    def __init__(self, pmf_z):
        # pmf_z ????????
        self.pmf_z = pmf_z
        
        # pmf_zb ???????????
        self.pmf_zb = BiasPmf(pmf_z)
        
        # pmf_y ????????
        self.pmf_y = PmfOfWaitTime(self.pmf_zb)
        
        # pmf_y ????????,?????????
        self.pmf_x = self.pmf_y.Copy()

def RemoveNegatives(pmf):
    for val in pmf.Values():
        if val < 0:
            pmf.Remove(val) # ? dictionary ??? key ??
    pmf.Normalize()


    
def PredictWaitTime(pmf_zb, pmf_x):
    # ???????? Pmf.__sub__ ???
    # Pmf.__sub__ ????? zb ? x ???????,????????????
    pmf_y = pmf_zb - pmf_x
    
    # pmf_y ??????????????? 5 ??,??????? 5 ??????(???????)
    # RemoveNegatives ???? pmf_y ??????????????
    RemoveNegatives(pmf_y)
    return pmf_y



class Elapsed(tb.Suite):
    
    def Likelihood(self, data, hypo):
        lam, num_passengers = data # ?????(unpacking)
        time_elapsed = hypo
        # ?????? Poisson ??
        # ???????????? num_passengers ?????
        like = tb.EvalPoissonPmf(num_passengers, time_elapsed*lam) 
        #print(data, hypo, like)
        return like
    

class ElapsedTimeEstimator(object):
    
    def __init__(self, wtc, lam, num_passengers):
        # wtc: ??? WaitTimeCalculator ??
        # lam: ??????(?/?)
        # num_passengers: ?????
        
        # x ?????,???? WaitTimeCalculator ? pmf_x
        self.prior_x = Elapsed(wtc.pmf_x)
        
        # x ?????
        self.post_x = self.prior_x.Copy()
        
        # ??????? (??????, ???????) ???
        self.post_x.Update((lam, num_passengers)) # ?? Elapsed ??? Likelihood ??
        
        # y = zb - x
        self.pmf_y = PredictWaitTime(wtc.pmf_zb, self.post_x)

        
class ArrivalRate(tb.Suite):
    
    def Likelihood(self, data, hypo):
        lam = hypo
        y, k = data
        like = tb.EvalPoissonPmf(k, lam*y)
        return like        

    
class ArrivalRateEstimator(object):
    
    def __init__(self, passenger_data):
        low, high = 0.1, 5
        n = 51
        
        # lam ??????? 
        hypos = np.linspace(low, high, n)
        
        self.prior_lam = ArrivalRate(hypos)
        
        self.post_lam = self.prior_lam.Copy()
        
        # ?????????,?????? post_lam
        for k1, y, k2 in passenger_data:
            self.post_lam.Update((y, k2))

  
gaps_min = np.array(OBSERVED_GAP_TIMES)/60.0
z = TruthGaps(gaps_min)        
wait_time = WaitTimeCalculator(z)

# lam ????? = ?? 2.0/60 ?
#ete = ElapsedTimeEstimator(wait_time, lam=2.0/60, num_passengers=15)
are = ArrivalRateEstimator(passenger_data) 

???????????????,??????????:

1. ??????????????????(??? ?)
1. ??????????
1. ????????,??????????
1. ??????????,????????????

?? 1 ? 2 ?????????? WaitMixtureEstimator ?????? 3 ? 4?

In [ ]:
class WaitMixtureEstimator(object):
    
    def __init__(self, wtc, are, num_passengers=15):
        self.metapmf = tb.Pmf()
        
        for lam, prob in sorted(are.post_lam.Items()):
            # ??? lam ?????????????????
            ete = ElapsedTimeEstimator(wtc, lam=lam, num_passengers=num_passengers)
            
            # ete.pmf_y ???? lam ???????
            self.metapmf.Set(ete.pmf_y, prob)
            
        self.mixture = tb.MakeMixture(self.metapmf)

# wait_time ? WaitTimeCalculator ???? zb ??
# are ? ArrivalTimeEstimator ???? lam ??
wme = WaitMixtureEstimator(wait_time, are)

In [ ]:
for pmf, prob in wme.metapmf.Items():
    plot_dist(pmf.MakeCdf(), c='c')

plot_dist(wme.mixture.MakeCdf(), c='r')
plt.title("CDF")
plt.show()

????? lam ?? CDF ????????????? lam ? CDF?????? CDF ?????

?????????? lam ???????,????????,???????????????

????,???????????,??????????;????????????????????

???????,lam ??????????????,??????????